## Extracting search events from logs

In [15]:
from driver import assign_topic_number
import glob
from tqdm.notebook import tqdm
import json
import pandas as pd

event_mapping = {
    "TaskStarted": "TS",
    "taskQuestionHelpClicked": "TH", 
    "queryBoxFocused": "QF", 
    "querySubmitted": "QS", # submitted user formulated query
    "searchResultGenerated": "SG", # serp generated
    "searchNoResults": "N_SG", # no SERP generated
    "clickedDidYouMeanSuggestion": "QC", # submitted SE corrected query
    "resultExposureStarted": "RV", # SERP result viewing
    "clickedResult": "RC", # SERP result clicked
    "customBackButtonClicked": "BC",
    "OpenedAnswerBox": "AW", # answer writing
    "TaskEnded": "TE",
}

event_sequences = []
relevance_df = pd.read_csv("Data/relevance.csv")

for file_name in tqdm(glob.glob("Data/3rd/*_FULL.log"), total=len(glob.glob("Data/3rd/*_FULL.log"))):
    with open(file_name, 'r') as f:
        log_entries = [json.loads(x) for x in f.readlines()]
    
    for entry in log_entries:
        if entry["type"] in list(event_mapping.keys()):
            uid = int(entry["uid"])
            topic_num = assign_topic_number(uid, entry["actual_topic_number"])
            event_type = entry["type"]
            event_key = event_mapping[event_type]
            event_time = entry["timestamp"]
            if (event_key == "RV") or (event_key == "RC"):
                url = entry["url"]
                relevant_resources = relevance_df.loc[relevance_df["topic_number"]==topic_num]["doc"].tolist()
                if url in relevant_resources:
                    event_key = "R_" + event_key
                else:
                    event_key = "NR_" + event_key
            if (event_key == "SG") and (int(entry["rank"]) > 1):
                continue
            event_sequences.append([uid, topic_num, event_type, event_key, event_time])

for file_name in tqdm(glob.glob("Data/4th/*_FULL.log"), total=len(glob.glob("Data/4th/*_FULL.log"))):
    with open(file_name, 'r') as f:
        log_entries = [json.loads(x) for x in f.readlines()]
    
    for entry in log_entries:
        if entry["type"] in list(event_mapping.keys()):
            uid = int(entry["uid"])
            topic_num = assign_topic_number(uid, entry["actual_topic_number"])
            event_type = entry["type"]
            event_key = event_mapping[event_type]
            event_time = entry["timestamp"]
            if (event_key == "RV") or (event_key == "RC"):
                url = entry["url"]
                relevant_resources = relevance_df.loc[relevance_df["topic_number"]==topic_num]["doc"].tolist()
                if url in relevant_resources:
                    event_key = "R_" + event_key
                else:
                    event_key = "NR_" + event_key
            if (event_key == "SG") and (int(entry["rank"]) > 1):
                continue
            event_sequences.append([uid, topic_num, event_type, event_key, event_time])

event_sequences_df = pd.DataFrame(event_sequences, columns=["uid", "topic_number", "event_name", "event_key", "timestamp"])   
event_sequences_df["timestamp"] = pd.to_datetime(event_sequences_df["timestamp"])   
event_sequences_df.to_csv("Data/event_sequences_df.csv", index=False, encoding="utf8")

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

In [20]:
event_sequences_df[event_sequences_df["event_key"]=="NR_RV"].head()

,uid,topic_number,event_name,event_key,timestamp
10,17,g3t3,resultExposureStarted,NR_RV,2026-05-19 14:51:04.466
13,17,g3t3,resultExposureStarted,NR_RV,2026-05-19 14:51:07.093
14,17,g3t3,resultExposureStarted,NR_RV,2026-05-19 14:51:08.610
15,17,g3t3,resultExposureStarted,NR_RV,2026-05-19 14:51:08.962
16,17,g3t3,resultExposureStarted,NR_RV,2026-05-19 14:51:13.155


In [11]:
event_sequences_df["uid"].unique()

array([17, 18,  1,  2,  3,  4,  5,  6,  7,  8, 10, 11, 12, 13, 14, 15, 16,
        9])

## Identifying common sequence patterns

In [17]:
import pandas as pd

event_sequences_df = pd.read_csv("Data/event_sequences_df.csv")
final_answer_df = pd.read_csv("Data/final_answer_df.csv")

In [18]:
from gsppy.gsp import GSP
from prefixspan import PrefixSpan


# successful_sessions = final_answer_df.loc[final_answer_df["answer_match_ratio"]==1.0]
# failed_sessions = final_answer_df.loc[final_answer_df["answer_match_ratio"]<1.0]

successful_transactions = []
failed_transactions = []

for _, row in final_answer_df.iterrows():
    uid = row["uid"]
    topic_num = row["topic_number"]
    session_actions = event_sequences_df.loc[(event_sequences_df["uid"]==uid) & (event_sequences_df["topic_number"]==topic_num)]
    if row["answer_match_ratio"] == 1.0:
        successful_transactions.append(session_actions["event_key"].tolist())
    else:
        failed_transactions.append(session_actions["event_key"].tolist())

# transactions = [
#     ["Bread", "Milk"],
#     ["Bread", "Diaper", "Beer", "Eggs"],
#     ["Milk", "Diaper", "Beer", "Coke"],
# ]

# patterns = GSP(successful_transactions).search(min_support=0.3, verbose=True)
# for level, freq_patterns in enumerate(patterns, start=1):
#     print(f"Level {level}: {freq_patterns}")

ps = PrefixSpan(successful_transactions)
patterns = ps.frequent(0.3*len(successful_transactions), closed=True) # all closed patterns occuring even once
patterns = sorted(patterns, reverse=True)
for sup, patt in patterns:
    print(f"{sup}: {' -> '.join(patt)}")

36: TS -> QF -> QS -> SG -> R_RV -> TE
36: TS -> QF -> QS -> SG -> R_RV -> AW -> TE
36: TS -> QF -> QS -> SG -> AW -> TE
35: TS -> QF -> QS -> SG -> R_RV -> R_RV -> TE
35: TS -> QF -> QS -> SG -> R_RV -> R_RV -> AW -> TE
31: TS -> TH -> AW -> TE
30: TS -> TH -> QF -> QS -> SG -> R_RV -> TE
30: TS -> TH -> QF -> QS -> SG -> R_RV -> AW -> TE
30: TS -> TH -> QF -> QS -> SG -> AW -> TE
29: TS -> TH -> QF -> QS -> SG -> R_RV -> R_RV -> TE
29: TS -> TH -> QF -> QS -> SG -> R_RV -> R_RV -> AW -> TE
29: TS -> QF -> TH -> AW -> TE
29: TS -> QF -> QS -> SG -> R_RV -> R_RV -> R_RV -> TE
29: TS -> QF -> QS -> SG -> R_RV -> R_RV -> R_RV -> AW -> TE
29: TS -> QF -> QS -> SG -> R_RV -> R_RC -> AW -> TE
29: TS -> QF -> QS -> SG -> R_RC -> AW -> TE
29: TS -> QF -> QF -> QS -> SG -> R_RV -> TE
29: TS -> QF -> QF -> QS -> SG -> R_RV -> AW -> TE
29: TS -> QF -> QF -> QS -> SG -> AW -> TE
29: TS -> QF -> QF -> AW -> TE
28: TS -> QF -> TH -> QF -> QS -> SG -> R_RV -> TE
28: TS -> QF -> TH -> QF -> QS -> SG 

In [19]:
ps = PrefixSpan(failed_transactions)
patterns = ps.frequent(0.3*len(failed_transactions), closed=True) # all closed patterns occuring even once
patterns = sorted(patterns, reverse=True)
for sup, patt in patterns:
    print(f"{sup}: {' -> '.join(patt)}")

18: TS -> TH -> QS -> SG -> R_RV -> AW -> TE
18: TS -> TH -> QF -> R_RV -> AW -> TE
18: TS -> TH -> QF -> QS -> SG -> R_RV -> AW -> TE
18: TS -> TH -> QF -> AW -> TE
17: TS -> TH -> TH -> QS -> SG -> R_RV -> AW -> TE
17: TS -> TH -> TH -> QF -> R_RV -> AW -> TE
17: TS -> TH -> TH -> QF -> QS -> SG -> R_RV -> AW -> TE
17: TS -> TH -> TH -> QF -> AW -> TE
17: TS -> TH -> QS -> SG -> R_RV -> R_RV -> AW -> TE
17: TS -> TH -> QF -> QS -> SG -> R_RV -> R_RV -> AW -> TE
17: TS -> TH -> QF -> QF -> R_RV -> AW -> TE
17: TS -> TH -> QF -> QF -> QS -> SG -> R_RV -> AW -> TE
17: TS -> TH -> QF -> QF -> AW -> TE
16: TS -> TH -> TH -> QS -> SG -> R_RV -> R_RV -> R_RV -> R_RV -> AW -> TE
16: TS -> TH -> TH -> QS -> SG -> R_RV -> R_RV -> R_RV -> AW -> TE
16: TS -> TH -> TH -> QS -> SG -> R_RV -> R_RV -> AW -> TE
16: TS -> TH -> TH -> QF -> QS -> SG -> R_RV -> R_RV -> R_RV -> R_RV -> AW -> TE
16: TS -> TH -> TH -> QF -> QS -> SG -> R_RV -> R_RV -> R_RV -> AW -> TE
16: TS -> TH -> TH -> QF -> QS -> SG -